### 1 - Extract e diagnóstico (tipos, ausentes, duplicatas, categorias)

Vamos carregar o arquivo de exames laboratoriais e fazer um diagnóstico inicial, verificando: o formato dos dados (shape), os tipos de cada coluna (dtypes), a 
quantidade de valores ausentes por coluna, a quantidade de linhas duplicadas, e a distribuição das categorias (sexo, prioridade, laboratorio, tipo_exame) para 
identificar inconsistências de grafia antes de qualquer correção.

In [1]:
import pandas as pd
import numpy as np

ex = pd.read_csv("dados/exames_laboratoriais.csv")

print("Formato:", ex.shape)
print("\nTipos de cada coluna:")
print(ex.dtypes)
print("\nValores ausentes por coluna:")
print(ex.isna().sum()[lambda s: s > 0])
print("\nLinhas duplicadas:", ex.duplicated().sum())
print("\nCategorias:")
for col in ["sexo", "prioridade", "laboratorio", "tipo_exame"]:
    print(f"\n{col} →", ex[col].value_counts(dropna=False).to_dict())

Formato: (155, 10)

Tipos de cada coluna:
id_exame        object
id_paciente     object
data_coleta     object
sexo            object
idade          float64
tipo_exame      object
resultado       object
unidade         object
laboratorio     object
prioridade      object
dtype: object

Valores ausentes por coluna:
idade    7
dtype: int64

Linhas duplicadas: 5

Categorias:

sexo → {'M': 72, 'F': 69, 'masculino': 10, 'Feminino': 4}

prioridade → {'Rotina': 97, 'Urgente': 36, 'ROTINA': 11, 'Emergência': 9, 'URGENTE': 1, 'EMERGÊNCIA': 1}

laboratorio → {'LabSaúde RN': 53, 'Laboratório Central': 49, 'LabNatal': 42, ' labnatal ': 6, ' labsaúde rn ': 5}

tipo_exame → {'Colesterol total': 46, 'Glicemia': 46, 'Creatinina': 39, 'Hemoglobina': 24}


### 2 - Remoção de duplicatas

Vamos remover as 5 linhas duplicadas identificadas no diagnóstico, resetar o índice e conferir que id_exame (o identificador único de cada exame) ficou sem repetições.

In [2]:
antes = len(ex)
ex = ex.drop_duplicates().reset_index(drop=True)
print(f"Linhas antes: {antes} | depois: {len(ex)}")
print("id_exame é único?", ex["id_exame"].is_unique)

Linhas antes: 155 | depois: 150
id_exame é único? True


### 3 - Transformação de tipos

Vamos converter data_coleta (que vem em dois formatos, ISO e brasileiro) para o tipo datetime, e resultado (texto com vírgula decimal, ex: "84,2") para float, para que 
ambas as colunas possam ser usadas em cálculos e comparações.

In [3]:
# data_coleta: dois formatos misturados (igual à Aula 09/10)
iso = pd.to_datetime(ex["data_coleta"], format="%Y-%m-%d", errors="coerce")
br = pd.to_datetime(ex["data_coleta"], format="%d/%m/%Y", errors="coerce")
ex["data_coleta"] = iso.fillna(br)

print("Datas não convertidas:", ex["data_coleta"].isna().sum())
print("Período:", ex["data_coleta"].min().date(), "a", ex["data_coleta"].max().date())

# resultado: texto com vírgula decimal (e o código 999, que trataremos na Seção 5)
ex["resultado"] = ex["resultado"].str.replace(",", ".", regex=False).astype(float)

print("\nTipo de 'resultado':", ex["resultado"].dtype)
print(ex["resultado"].describe().round(2))

Datas não convertidas: 0
Período: 2026-07-02 a 2026-09-28

Tipo de 'resultado': float64
count    150.00
mean     112.05
std      167.42
min        0.47
25%       12.42
50%       87.05
75%      171.48
max      999.00
Name: resultado, dtype: float64


### 4 - Padronização das categorias

Vamos padronizar as três colunas categóricas que apareceram com grafias inconsistentes no diagnóstico: sexo (M, F, masculino, Feminino -> F/M), prioridade (ROTINA, URGENTE,
EMERGÊNCIA -> Rotina/Urgente/Emergência) e laboratorio (espaços extras e minúsculas -> nomes oficiais LabNatal, LabSaúde RN e Laboratório Central).

In [4]:
# sexo: primeira letra maiúscula (masculino -> M, Feminino -> F)
ex["sexo"] = ex["sexo"].str.strip().str[0].str.upper()

# prioridade: tira espaços e deixa só a primeira letra maiúscula (ROTINA -> Rotina)
ex["prioridade"] = ex["prioridade"].str.strip().str.capitalize()

# laboratorio: tira espaços, uniformiza em minúsculas e mapeia para o nome oficial
nomes_oficiais = {
    "labnatal": "LabNatal",
    "labsaúde rn": "LabSaúde RN",
    "laboratório central": "Laboratório Central",
}
ex["laboratorio"] = ex["laboratorio"].str.strip().str.lower().map(nomes_oficiais)

for col in ["sexo", "prioridade", "laboratorio"]:
    print(f"{col} →", ex[col].value_counts(dropna=False).to_dict())

sexo → {'M': 80, 'F': 70}
prioridade → {'Rotina': 105, 'Urgente': 35, 'Emergência': 10}
laboratorio → {'LabSaúde RN': 58, 'Laboratório Central': 48, 'LabNatal': 44}
